In [1]:
import os

def setting_up_proxy(proxy=None, proxy_type='http', verbose=True):
    supported_proxy_types = ['http', 'https', 'socks4', 'socks5', 'all']
    assert proxy_type in supported_proxy_types, f"proxy type {repr(proxy_type)} not supported, only support {supported_proxy_types}"
    if proxy is None:
        proxy = os.environ.get(f'{proxy_type}_proxy')
    if proxy is None:
        return
    if verbose:
        print(f'setting up proxy {repr(proxy)} for {repr(proxy_type)}')
    os.environ[f'{proxy_type}_proxy'] = proxy


# default_proxy_config = {
#     'http': 'http://127.0.0.1:7890',
#     'https': 'http://127.0.0.1:7890',
#     'all': 'socks5://127.0.0.1:7890',
# }


default_proxy_config = {
    'http': 'http://10.176.52.116:7890',
    'https': 'http://10.176.52.116:7890',
    'all': 'socks5://10.176.52.116:7890',
}


def setting_up_proxy_from_config(proxy_config=default_proxy_config, verbose=True):
    for proxy_type, proxy_url in proxy_config.items():
        setting_up_proxy(proxy=proxy_url, proxy_type=proxy_type, verbose=verbose)
    print()


# setting_up_proxy_from_config()

In [2]:
import requests
import os
import json


resp = requests.get('https://xiageba.com/music/108096', proxies=default_proxy_config)
print(resp)  # sometimes 429: Too Many Requests
# print(resp.content)
print(resp.headers['content-type'])

with open('examples/108096.html', 'wb') as f:
    f.write(resp.content)

<Response [200]>
text/html;text/html


In [30]:
from RFC.utils.parse import (
    get_html_soup,
    parse,
)


def parse_kimi_links(resp):
    parse_config = {
        'metas': {
            ('attr', 'div', 'class', 'd-flex flex-column justify-content-center', None): {
                ('type', 'div', None): {
                    ('result', 'text', (('return_as_list', True),), None): {},
                }
            }
        },
        'links': {
            ('attr', 'div', 'class', 'down-item', None): {
                ('result', 'text', (('return_as_list', True),), None): {},
            }
        },
        'tags': {
            ('attr', 'div', 'class', 'd-flex flex-wrap', 0): {
                ('type', 'a', None): {
                    ('result', 'text', (('return_as_list', True),), None): {},
                }
            }
        },
        'lyrics': {
            ('attr', 'div', 'class', 'border-top mt-3 pt-3', None): {
                ('attr', 'div', 'class', 'mt-2', 0): {
                    ('result', 'text', (('sep', '\n'), ('return_as_list', True),), None): {},
                }
            }
        }
        
    }
    links = parse(get_html_soup(resp.content), parse_config['links'])
    _links = []
    for link in links:
        if len(link) & 1:
            link.append('')
        for i, part in enumerate(link[::2]):
            if part.endswith(':'):
                link[i*2] = part[:-1]
        _links.append({k: v for k, v in zip(link[::2], link[1::2])})
        
    metas = parse(get_html_soup(resp.content), parse_config['metas'])
    tags = parse(get_html_soup(resp.content), parse_config['tags'])
    lyrics = parse(get_html_soup(resp.content), parse_config['lyrics'], debug=True)
    # we delay the format checking to the post-processing stage
    
    results = {
        'metadata': metas,
        'tags': tags,
        'links': _links,
        'lyrics': lyrics,
    }
    
    return results

info = parse_kimi_links(resp)
import json
print(json.dumps(info, indent=4, ensure_ascii=False))

<!DOCTYPEhtml><htmllang="zh-CN"><head><metacharset="utf-8"/><metacontent="width=device-width,initial...
├── <divclass="border-topmt-3pt-3"><inputid="oInput"style="display:none;"type="text"/><divclass="down-it...
│   └── None
├── <divclass="border-topmt-3pt-3"><divclass="d-flexjustify-content-between"><spanclass="fs-10fw-bold">歌...
│   └── <divclass="mt-2">如果你出征<br/>我以酒相送<br/>带三分醉意去驰骋纵横<br/>我要在东边挂一道彩虹<br/>装点你那闪亮的行程<br/>如果你称雄就该做先锋<br/>带七分豪...
│       └── ['如果你出征','我以酒相送','带三分醉意去驰骋纵横','我要在东边挂一道彩虹','装点你那闪亮的行程','如果你称雄就该做先锋','带七分豪情去立业建功','我要在西边采一抹火红','渲染你那凯...
└── <divclass="border-topmt-3pt-3"><divclass="d-flexjustify-content-between"><divclass="d-flexalign-item...
    └── None
{
    "metadata": [
        [
            "一吻赏英雄"
        ],
        [
            "可晴"
        ],
        [
            "更新：2023-12-2715:31"
        ]
    ],
    "tags": [
        [
            "一吻赏英雄"
        ],
        [
            "可晴"
        ],
        [
            "FLAC/MP3-320K"
        ],
        [
       